In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score


In [ ]:
# 데이터 불러오기
data_path = Path('A_DeviceMotion_data 복사본/HAR_total.pkl')
har_total = pd.read_pickle(data_path).copy()

# magnitude가 없으면 생성
if 'magrotationRate' not in har_total.columns:
    har_total['magrotationRate'] = np.sqrt(
        har_total['rotationRate.x'] ** 2
        + har_total['rotationRate.y'] ** 2
        + har_total['rotationRate.z'] ** 2
    )

if 'maguserAcceleration' not in har_total.columns:
    har_total['maguserAcceleration'] = np.sqrt(
        har_total['userAcceleration.x'] ** 2
        + har_total['userAcceleration.y'] ** 2
        + har_total['userAcceleration.z'] ** 2
    )

print(har_total.shape)
har_total[['exp_no', 'id', 'activity', 'magrotationRate', 'maguserAcceleration']].head()


In [ ]:
# 피크 검출 함수
def peak_table(signal, threshold=4.0, top_n=None):
    values = np.asarray(signal, dtype=float)
    peak_idx, props = find_peaks(values, height=threshold)
    peaks = pd.DataFrame(
        {
            'peak_height': props.get('peak_heights', np.array([], dtype=float)),
            'peak_index': peak_idx,
        }
    ).sort_values('peak_height', ascending=False)
    return peaks.head(top_n) if top_n else peaks


# 파고율 함수
def crest_factor(x):
    values = np.asarray(x, dtype=float)
    rms = np.sqrt(np.mean(values ** 2))
    return 0.0 if rms == 0 else float(np.max(np.abs(values)) / rms)


# 피크 통계 함수
def peak_stats(peaks):
    peak_positions = np.sort(peaks['peak_index'].to_numpy()) if len(peaks) else np.array([])
    peak_values = peaks['peak_height'].to_numpy(dtype=float) if len(peaks) else np.array([])
    intervals = np.diff(peak_positions) if len(peaks) >= 2 else np.array([])
    return {
        'f_n': int(len(peaks)),
        'p_interval': float(intervals.mean()) if len(intervals) else 0.0,
        'p_interval_std': float(intervals.std(ddof=1)) if len(intervals) >= 2 else 0.0,
        'p_mean': float(peak_values.mean()) if len(peak_values) else 0.0,
        'p_max': float(peak_values.max()) if len(peak_values) else 0.0,
        'p_min': float(peak_values.min()) if len(peak_values) else 0.0,
        'p_std': float(peak_values.std(ddof=1)) if len(peak_values) >= 2 else 0.0,
    }


In [ ]:
# 시퀀스별 피크 특징 만들기
feature_rows = []

for (exp_no, sid, activity), group in har_total.groupby(['exp_no', 'id', 'activity'], sort=True):
    signal = group['magrotationRate'].to_numpy(dtype=float)
    accel = group['maguserAcceleration'].to_numpy(dtype=float)
    peaks = peak_table(signal, threshold=4.0)
    row = peak_stats(peaks)
    row.update(
        {
            'cfR': crest_factor(signal),
            'cfA': crest_factor(accel),
        }
    )
    feature_rows.append(
        {
            'sequence_key': f'exp_{exp_no}_id_{sid}_{activity}',
            'exp_no': exp_no,
            'id': sid,
            'activity': activity,
            **row,
        }
    )

peak_final = pd.DataFrame(feature_rows)[
    [
        'sequence_key', 'exp_no', 'id', 'activity',
        'f_n', 'p_interval', 'p_interval_std',
        'p_mean', 'p_max', 'p_min', 'p_std',
        'cfR', 'cfA',
    ]
]

print(peak_final.shape)
peak_final.head()


In [ ]:
# 예시 시퀀스 선택
example_info = peak_final.iloc[0]
example = har_total[
    (har_total['exp_no'] == example_info['exp_no'])
    & (har_total['id'] == example_info['id'])
    & (har_total['activity'] == example_info['activity'])
].copy()

print(example_info[['sequence_key', 'activity']])
example.head()


In [ ]:
# 예시 시퀀스 그리기
plt.figure(figsize=(10, 4))
plt.plot(example['magrotationRate'].to_numpy(), color='steelblue')
plt.xlabel('Sample index')
plt.ylabel('magrotationRate')
plt.title(str(example_info['sequence_key']))
plt.show()


In [ ]:
# 예시 시퀀스 피크 표시
example_peaks = peak_table(example['magrotationRate'], threshold=5.0)

plt.figure(figsize=(10, 4))
plt.plot(example['magrotationRate'].to_numpy(), color='steelblue')
plt.scatter(example_peaks['peak_index'], example_peaks['peak_height'], color='crimson', s=18)
plt.xlabel('Sample index')
plt.ylabel('magrotationRate')
plt.title('Detected peaks')
plt.show()

example_peaks.head()


In [ ]:
# 분류용 데이터 준비
feature_cols = [
    'f_n', 'p_interval', 'p_interval_std',
    'p_mean', 'p_max', 'p_min', 'p_std',
    'cfR', 'cfA',
]

X = peak_final[feature_cols].fillna(0.0)
y = peak_final['activity']

model = RandomForestClassifier(n_estimators=300, random_state=42)
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
accuracy = cross_val_score(model, X, y, cv=cv, scoring='accuracy')

metrics = pd.DataFrame(
    {
        'metric': ['cv_accuracy_mean', 'cv_accuracy_std', 'num_sequences', 'num_features'],
        'value': [accuracy.mean(), accuracy.std(ddof=1), len(peak_final), len(feature_cols)],
    }
)

print('mean accuracy:', round(accuracy.mean(), 6))
print('std:', round(accuracy.std(ddof=1), 6))
metrics


In [ ]:
# 연습문제: 통계 특징 + 피크 특징 결합
stat_final = (
    har_total.groupby(['exp_no', 'id', 'activity'], sort=True)
    .agg(
        rot_mean=('magrotationRate', 'mean'),
        rot_min=('magrotationRate', 'min'),
        rot_max=('magrotationRate', 'max'),
        rot_std=('magrotationRate', 'std'),
        rot_skew=('magrotationRate', 'skew'),
        acc_mean=('maguserAcceleration', 'mean'),
        acc_min=('maguserAcceleration', 'min'),
        acc_max=('maguserAcceleration', 'max'),
        acc_std=('maguserAcceleration', 'std'),
        acc_skew=('maguserAcceleration', 'skew'),
    )
    .reset_index()
)

combined_final = peak_final.merge(stat_final, on=['exp_no', 'id', 'activity'], how='left')

combined_feature_cols = feature_cols + [
    'rot_mean', 'rot_min', 'rot_max', 'rot_std', 'rot_skew',
    'acc_mean', 'acc_min', 'acc_max', 'acc_std', 'acc_skew',
]

X_combined = combined_final[combined_feature_cols].fillna(0.0)
y_combined = combined_final['activity']

combined_model = RandomForestClassifier(n_estimators=300, random_state=42)
combined_accuracy = cross_val_score(combined_model, X_combined, y_combined, cv=cv, scoring='accuracy')

comparison = pd.DataFrame(
    {
        'model': ['peak_only', 'peak_plus_stats'],
        'mean_accuracy': [accuracy.mean(), combined_accuracy.mean()],
        'std_accuracy': [accuracy.std(ddof=1), combined_accuracy.std(ddof=1)],
        'num_features': [len(feature_cols), len(combined_feature_cols)],
    }
)

print('peak + stats mean accuracy:', round(combined_accuracy.mean(), 6))
print('peak + stats std:', round(combined_accuracy.std(ddof=1), 6))
comparison
